[//]: # (cr:doc name='chapter_c01_publish_definitions' id=d3894d72)
# Chapter c01: Publish Definition Tables (Causal Track)

Reads `*.yaml` from `PLAYBOOKS_DIR/` (the playbook catalog) and `PLAYBOOKS_DIR/policies/` (the operational policies) and writes them to the corresponding Delta tables. Full overwrite on every publish — these are small + authoritative from YAML.

Run this notebook **before** `c02_archetype_derivation` whenever the playbook YAMLs change.


In [ ]:
# @cr:code name='init_progress' id=735e4015
from customer_retention.analysis.notebook_progress import accept_workflow_params, track_and_export_previous

accept_workflow_params()
track_and_export_previous("c01_publish_definitions.ipynb")
# --- cr:profiler ---
if __import__('os').environ.get("CR_BATCH_EXECUTION") == "1":
    import json as _j
    import os as _os
    import re as _r
    _cr_nb = _os.path.splitext(_os.path.basename(_os.environ.get("PAPERMILL_OUTPUT_PATH", "")))[0]
    if _cr_nb:
        _cr_mp = _os.path.join(_os.getcwd(), f".cr_cell_metrics_{_cr_nb}.jsonl")
        open(_cr_mp, 'w').close()
        _cr_re = _r.compile(r"^#\s*@cr:\w+\s+name='([^']+)'\s+id=(\w+)")
        def _cr_jc():
            return -1
        try:
            _s = __import__('pyspark.sql', fromlist=['SparkSession']).SparkSession.getActiveSession()
            if _s:
                def _cr_jc():  # noqa: F811
                    return _s._jsc.sc().dagScheduler().nextJobId().get()
        except Exception:
            pass
        def _cr_pre(info):
            info._cr_sj = _cr_jc()
        def _cr_post(r):
            sj = getattr(r.info, '_cr_sj', -1)
            sa = _cr_jc()
            m = _cr_re.match((r.info.raw_cell or '').split('\n')[0])
            if m:
                with open(_cr_mp, 'a') as f:
                    f.write(_j.dumps({"cell_name": m.group(1), "cell_id": m.group(2),
                                      "spark_jobs": (sa - sj) if sj >= 0 and sa >= 0 else None}) + '\n')
        get_ipython().events.register('pre_run_cell', _cr_pre)
        get_ipython().events.register('post_run_cell', _cr_post)
# --- /cr:profiler ---


[//]: # (cr:doc name='c01_configuration' id=a30b9bd2)
## Configuration

The cell below is the only place you should need to edit. Every value here is read by the publish cell and the pipeline runner below — nothing is hardcoded inside the algorithmic cells.

- **`SKIP_PUBLISH_DEFINITIONS`** — short-circuit the publish step (e.g. when the YAMLs are unchanged since the last run).
- **`RISK_TIER_HIGH_THRESHOLD` / `RISK_TIER_MEDIUM_THRESHOLD`** — risk tier cutoffs mirrored into `decision_policy` so historical assignments can be reconstructed from the policy version in force at scoring time. The publish step writes these as the on-disk default; if a YAML row in `decision_policy.yaml` already specifies them, the YAML wins.
- **`SKIP_PIPELINE_RUN`** — skip invoking the generated pipeline (e.g. when predictions are already fresh).
- **`PIPELINE_DIR`** — directory containing the generated pipeline scripts produced by `exploration_notebooks/10_spec_generation.ipynb`. Leave as `None` to resolve to `/Workspace/{workspace_path}/generated_pipelines/databricks/{pipeline_name}` via `get_workspace_path()` + `ScoringConfig`. Override if the scripts live elsewhere.
- **`PIPELINE_STAGES`** — ordered stage subdirectories to execute. Includes `scoring` so the `predictions` Delta table is populated before `c05_snapshot_and_dashboard` runs. (The standalone `c04_batch_inference` notebook is an alternative when you only want to refresh scoring without re-running the full training pipeline.)
- **`PIPELINE_STAGE_TIMEOUT_SECONDS`** — per-notebook timeout passed to `dbutils.notebook.run()`.


In [ ]:
# @cr:config name='configuration' id=81c0d9bc
SKIP_PUBLISH_DEFINITIONS = False

RISK_TIER_HIGH_THRESHOLD = 0.6
RISK_TIER_MEDIUM_THRESHOLD = 0.3

SKIP_PIPELINE_RUN = False

PIPELINE_DIR = None
PIPELINE_STAGES = ["landing", "bronze", "silver", "gold", "training", "scoring"]
PIPELINE_STAGE_TIMEOUT_SECONDS = 3600


[//]: # (cr:doc name='c01_publish_definitions_setup' id=87180a11)
## 1.0 Setup

Resolves catalog / schema / model identifiers from `ScoringConfig` (reads the persisted Databricks init JSON on Databricks, or the local pipeline's `best_model_meta.json` for local runs). The composite-name-qualified gold features table name is derived here so the algorithmic cells stay free of path-construction logic.


In [ ]:
# @cr:code name='setup_and_resolve_model' id=fe8a0b5b
from customer_retention.core.compat.detection import get_spark_session, is_databricks
from customer_retention.core.config import get_playbooks_dir
from customer_retention.core.config.experiments import get_experiments_dir
from customer_retention.stages.scoring import ScoringConfig

spark = get_spark_session()
PLAYBOOKS_DIR = get_playbooks_dir()

if is_databricks():
    scoring_config = ScoringConfig.from_databricks()
    CATALOG = scoring_config.catalog
    SCHEMA = scoring_config.schema
else:
    scoring_config = ScoringConfig.from_local_config(get_experiments_dir())
    CATALOG = "local"
    SCHEMA = "local"

PLAYBOOK_CATALOG_FQN = f"{CATALOG}.{SCHEMA}.playbook_catalog"
PLAYBOOK_STEPS_FQN = f"{CATALOG}.{SCHEMA}.playbook_steps"
DECISION_POLICY_FQN = f"{CATALOG}.{SCHEMA}.decision_policy"
RESPONSE_SCHEMAS_FQN = f"{CATALOG}.{SCHEMA}.response_schemas"
VOCABULARIES_FQN = f"{CATALOG}.{SCHEMA}.vocabularies"

print(f"Resolved playbooks_dir: {PLAYBOOKS_DIR}")
print(f"Catalog/schema:         {CATALOG}.{SCHEMA}")


[//]: # (cr:doc name='c01_publish_section' id=ef0abc91)
## 1.1 Publish Definition Tables (YAML → Delta)


In [ ]:
# @cr:code name='publish_definition_tables' id=3de4ae53
from customer_retention.stages.causal.delta_writer import overwrite_table
from customer_retention.stages.causal.playbook_loader import load_playbooks_from_dir
from customer_retention.stages.causal.policy_loader import load_policies_from_dir
from customer_retention.stages.causal.schemas import (
    decision_policy_schema,
    playbook_catalog_schema,
    playbook_steps_schema,
    response_schemas_schema,
    vocabularies_schema,
)

if SKIP_PUBLISH_DEFINITIONS:
    print("SKIPPED: SKIP_PUBLISH_DEFINITIONS=True")
elif spark is None:
    print("SKIPPED: no active Spark session (Databricks-only cell)")
else:
    catalog_rows, step_rows = load_playbooks_from_dir(PLAYBOOKS_DIR)
    policies = load_policies_from_dir(PLAYBOOKS_DIR)

    decision_rows = policies.get("decision_policy", [])
    for row in decision_rows:
        if row.get("risk_tier_high_threshold") is None:
            row["risk_tier_high_threshold"] = RISK_TIER_HIGH_THRESHOLD
        if row.get("risk_tier_medium_threshold") is None:
            row["risk_tier_medium_threshold"] = RISK_TIER_MEDIUM_THRESHOLD

    overwrite_table(spark, catalog_rows, playbook_catalog_schema(), PLAYBOOK_CATALOG_FQN)
    overwrite_table(spark, step_rows, playbook_steps_schema(), PLAYBOOK_STEPS_FQN)
    overwrite_table(spark, decision_rows, decision_policy_schema(), DECISION_POLICY_FQN)
    overwrite_table(spark, policies.get("response_schemas", []), response_schemas_schema(), RESPONSE_SCHEMAS_FQN)
    overwrite_table(spark, policies.get("vocabularies", []), vocabularies_schema(), VOCABULARIES_FQN)
    print(
        f"Published {len(catalog_rows)} playbooks, {len(step_rows)} steps, "
        f"{len(decision_rows)} decision_policy rows"
    )


[//]: # (cr:doc name='c01_run_pipeline_section' id=440693a6)
## 1.2 Run the Generated Pipeline (s01 → s10)

Before `c02_archetype_derivation`, `c04_batch_inference`, and `c05_snapshot_and_dashboard` can run, three artifacts must exist on the cluster:

1. **Gold features table** — `{CATALOG}.{SCHEMA}.customer_features` registered as a feature table
2. **Registered `@production` model** — `models:/{MODEL_NAME}@production` (where `MODEL_NAME` is the 3-part Unity Catalog FQN written by training as `registered_model_name`)
3. **Predictions table** — `{CATALOG}.{SCHEMA}.predictions` (one row per scored customer)

The cells below invoke the generated pipeline scripts produced by `exploration_notebooks/10_spec_generation.ipynb`, in the order **landing → bronze → silver → gold → training → scoring**. Each stage runs as a Databricks notebook task via `dbutils.notebook.run()` — the same pattern NB10 uses for the training leg, extended here to include `scoring` (s10) so the `predictions` table is populated.

**Progress is captured per stage**: elapsed time, any JSON result returned via `dbutils.notebook.exit(...)`, and a rolling total. A failure in any stage raises immediately; nothing downstream runs. Set `SKIP_PIPELINE_RUN = True` in the configuration cell to bypass this block when predictions are already fresh.


In [ ]:
# @cr:code name='run_generated_pipeline' id=57f7e7dd
import json as _json
import time as _time
from pathlib import Path as _Path

from customer_retention.core.compat.detection import get_dbutils, get_spark_session
from customer_retention.core.config.experiments import get_workspace_path
from customer_retention.stages.scoring import ScoringConfig
from customer_retention.stages.scoring.pipeline_discovery import find_generated_pipeline_dir


def _resolve_default_pipeline_dir() -> _Path:
    workspace_path = get_workspace_path()
    if not workspace_path:
        raise RuntimeError(
            "Cannot resolve default PIPELINE_DIR: CR_WORKSPACE_PATH is not set. "
            "Either run databricks_init(workspace_path=...) first, or set PIPELINE_DIR "
            "in the configuration cell to an absolute workspace path."
        )
    scoring_config = ScoringConfig.from_databricks()
    return find_generated_pipeline_dir(_Path(f"/Workspace/{workspace_path}"), scoring_config)


_g = globals()
_skip = _g.get("SKIP_PIPELINE_RUN", False)
_pipeline_dir_cfg = _g.get("PIPELINE_DIR", None)
_stages = _g.get(
    "PIPELINE_STAGES",
    ["landing", "bronze", "silver", "gold", "training", "scoring"],
)
_timeout = _g.get("PIPELINE_STAGE_TIMEOUT_SECONDS", 3600)
_spark = _g.get("spark") or get_spark_session()

_pipeline_results = {}
_pipeline_errors = []

_dbutils = get_dbutils()

if _skip:
    print("SKIPPED: SKIP_PIPELINE_RUN=True")
elif _spark is None:
    print("SKIPPED: no active Spark session (Databricks-only cell)")
elif _dbutils is None:
    print("SKIPPED: dbutils unavailable (not running on Databricks)")
else:
    _pipeline_dir = _Path(_pipeline_dir_cfg) if _pipeline_dir_cfg else _resolve_default_pipeline_dir()
    if not _pipeline_dir.exists():
        raise FileNotFoundError(
            f"Generated pipeline directory not found at {_pipeline_dir}. "
            "Run exploration_notebooks/10_spec_generation.ipynb first to produce it, "
            "or set PIPELINE_DIR in the configuration cell."
        )

    print(f"PIPELINE_DIR: {_pipeline_dir}")
    print(f"STAGES:       {_stages}")
    print("=" * 70)

    _total_start = _time.time()
    for _stage in _stages:
        _stage_dir = _pipeline_dir / _stage
        if not _stage_dir.exists():
            print(f"[{_stage.upper():<9}] (no scripts in {_stage_dir.name}/ - skipping)")
            continue
        _notebooks = sorted(f.stem for f in _stage_dir.iterdir() if f.suffix == ".py")
        if not _notebooks:
            print(f"[{_stage.upper():<9}] (empty)")
            continue
        for _nb in _notebooks:
            _path = str(_stage_dir / _nb)
            _start = _time.time()
            print(f"[{_stage.upper():<9}] {_nb} ... ", end="", flush=True)
            try:
                _result = _dbutils.notebook.run(_path, _timeout, {})
                _elapsed = _time.time() - _start
                print(f"{_elapsed:>7.1f}s")
                if _result:
                    try:
                        _pipeline_results[_nb] = _json.loads(_result)
                    except (ValueError, TypeError):
                        _pipeline_results[_nb] = {"raw": _result}
            except Exception as _exc:
                _elapsed = _time.time() - _start
                print(f"FAILED after {_elapsed:.1f}s")
                _pipeline_errors.append({"stage": _stage, "notebook": _nb, "error": str(_exc)})
                raise

    print("=" * 70)
    print(f"Total elapsed: {_time.time() - _total_start:.1f}s")
    print(f"Notebooks with returned results: {len(_pipeline_results)}")


In [ ]:
# @cr:code name='pipeline_summary' id=ec3553ec
# Summary: surface training metrics from the training stage (if it returned JSON)
# and confirm the predictions Delta table is populated with a risk-tier distribution.

from customer_retention.core.compat.detection import get_spark_session
from customer_retention.stages.scoring import ScoringConfig

# Resolve upstream state defensively (see run_generated_pipeline cell comment).
_g = globals()
_skip = _g.get("SKIP_PIPELINE_RUN", False)
_spark = _g.get("spark") or get_spark_session()
_results = _g.get("_pipeline_results", {})
_catalog = _g.get("CATALOG")
_schema = _g.get("SCHEMA")
if _catalog is None or _schema is None:
    try:
        _sc = ScoringConfig.from_databricks()
        _catalog = _catalog or _sc.catalog
        _schema = _schema or _sc.schema
    except Exception:
        pass

if _skip:
    print("SKIPPED: SKIP_PIPELINE_RUN=True")
elif _spark is None:
    print("SKIPPED: no active Spark session")
elif not _catalog or not _schema:
    print("SKIPPED: CATALOG/SCHEMA not resolved (setup cell did not run)")
else:
    _predictions_fqn = f"{_catalog}.{_schema}.predictions"

    print("=" * 70)
    print("PIPELINE SUMMARY")
    print("=" * 70)

    _training_result = next(
        (v for k, v in _results.items() if "train" in k.lower() and isinstance(v, dict)),
        None,
    )
    if _training_result:
        print("\nTraining results:")
        _models = _training_result.get("models")
        if _models:
            print(f"  {'Model':<25} {'AUC':>8} {'PR-AUC':>8} {'F1':>8}")
            print(f"  {'-' * 25} {'-' * 8} {'-' * 8} {'-' * 8}")
            for _name, _metrics in _models.items():
                print(
                    f"  {_name:<25} {_metrics.get('roc_auc', 0):>8.4f} "
                    f"{_metrics.get('pr_auc', 0):>8.4f} {_metrics.get('f1', 0):>8.4f}"
                )
        _best = _training_result.get("best_model")
        if _best:
            print(f"  Best: {_best} (AUC={_training_result.get('best_roc_auc', 0):.4f})")

    try:
        _pred = _spark.table(_predictions_fqn)
        _total = _pred.count()
        print(f"\nPredictions table: {_predictions_fqn}")
        print(f"  rows: {_total:,}")
        if _total > 0:
            print("\n  Risk-tier distribution:")
            _pred.groupBy("risk_tier").count().orderBy("risk_tier").show(truncate=False)
            print("  Model URI(s) in this table:")
            _pred.select("model_uri").distinct().show(truncate=False)
            print("  Sample rows:")
            _pred.limit(10).show(truncate=False)
        else:
            print("  WARNING: predictions table is empty. Check scoring stage logs above.")
    except Exception as _exc:
        print(f"\nERROR reading {_predictions_fqn}: {_exc}")
        print("Pipeline may have failed before scoring. Inspect the output above.")


In [ ]:
# @cr:code name='release_stage_memory' id=54f197b0
from customer_retention.core.compat import release_stage_memory

release_stage_memory()
